# Spin precession and Rabi flopping

*quantum · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/spin_precession)

A spin-$\tfrac12$ in a magnetic field. The Hamiltonian is a contraction of the field vector with the Pauli tensor $\sigma_{kab}$, the state is evolved with the matrix exponential $U = \exp(-iH\Delta t)$, and the Bloch vector $s_k = \langle\psi\vert\sigma_k\vert\psi\rangle$ is one more einsum. A static field makes the spin precess at the Larmor frequency; a weak resonant transverse field slowly flips it (Rabi oscillation).

**Vector form**

$$i\,\partial_t|\psi\rangle = \hat H|\psi\rangle,\quad \hat H = -\tfrac{\gamma}{2}\,\mathbf B\cdot\boldsymbol\sigma,\quad \mathbf s = \langle\psi|\boldsymbol\sigma|\psi\rangle$$

**Index (tensor) form**

$$H_{ab} = -\tfrac{\gamma}{2}\,B_k\,\sigma_{kab},\qquad \psi_a(t+\Delta t) = U_{ab}\,\psi_b(t),\qquad s_k = \psi^*_a\,\sigma_{kab}\,\psi_b$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\sigma_{kab}$ | `pauli()` | (3, 2, 2) complex | Pauli tensor: $k \in \{x, y, z\}$ picks the matrix, $a, b$ are spinor indices |
| $B_k(t)$ | `field(t, B0, B1, w)` | (T, 3) | static $B_0$ along $z$ plus an optional $B_1\cos(\omega t)$ along $x$ |
| $H_{ab}$ | `einsum('k,kab->ab', B, sigma)` | (2, 2) | repeated $k$ ⇒ summed: $\mathbf B\cdot\boldsymbol\sigma$ as a $2\times 2$ matrix (batched over $t$ as tk,kab->tab) |
| $U_{ab}$ | `torch.linalg.matrix_exp(-1j * H * dt)` | (T, 2, 2) | one-step propagator, unitary |
| $s_k$ | `einsum('a,kab,b->k', psi.conj(), sigma, psi)` | (3,) | Bloch vector: $a$ and $b$ both summed |

Units $\hbar = 1$. With $B_1 = 0$ the spin precesses about $z$ at the Larmor frequency $\omega = \gamma B_0$, and $\lvert\mathbf s\rvert = 1$ forever because $U$ is unitary. With $B_1 > 0$ driven at $\omega \approx \gamma B_0$ the spin spirals from the north pole towards the south pole and back: in the frame rotating with the drive only the co-rotating half of the linear drive matters, giving the Rabi frequency $\Omega = \gamma B_1/2$ (rotating-wave approximation). Detune the drive and the flip becomes incomplete. The Pauli matrices obey $\sigma_i\sigma_j = \delta_{ij}I + i\,\varepsilon_{ijk}\sigma_k$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/spin_precession/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    sigma = tensors.pauli(dev)
    psi0 = tensors.tilted_spinor(params["theta0"], dev)

    times, psi, s = solve.solve(
        psi0,
        sigma,
        params["gamma"],
        params["B0"],
        params["B1"],
        params["drive_freq"],
        params["t_max"],
    )

    print(f"Larmor frequency gamma*B0 = {params['gamma'] * params['B0']:.3f}")
    print(f"Rabi frequency (resonant, RWA) gamma*B1/2 = {params['gamma'] * params['B1'] / 2:.3f}")
    print(f"min s_z = {s[:, 2].min():.4f}   max | |s| - 1 | = {(s.norm(dim=1) - 1).abs().max():.2e}")
    return plot.animate(times, s, out_dir / "spin_precession.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/spin_precession/equation.py
r"""Physics → tensor translation.

A spin-½ in a magnetic field (units ħ = 1):

    i \partial_t |\psi\rangle = \hat H |\psi\rangle,      \hat H = -\tfrac{\gamma}{2} \mathbf B \cdot \boldsymbol\sigma.

Stack the three Pauli matrices into one tensor \sigma_{kab} (k: x, y, z; a, b: spinor indices).
Then the dot product B·σ is a contraction over k:

    H_{ab} = -\tfrac{\gamma}{2} B_k \sigma_{kab}                     (k summed)

Over one short step the field is nearly constant, so

    \psi_a(t + \Delta t) = U_{ab} \psi_b(t),   U = e^{-i H \Delta t}     (b summed)

and the Bloch vector, the expectation of each Pauli matrix, sums over both spinor indices:

    s_k = \psi^*_a \sigma_{kab} \psi_b                                (a, b summed)

Its equation of motion is d\mathbf s/dt = \gamma\, \mathbf s \times \mathbf B: precession at \omega = \gamma B.
"""

import torch


def hamiltonian(B: torch.Tensor, sigma: torch.Tensor, gamma: float) -> torch.Tensor:
    r"""H_{tab} = -(\gamma/2) B_{tk} \sigma_{kab}.   B: (T, 3), sigma: (3, 2, 2) → (T, 2, 2)"""
    return -0.5 * gamma * torch.einsum("tk,kab->tab", B.to(sigma.dtype), sigma)


def propagator(H: torch.Tensor, dt: float) -> torch.Tensor:
    r"""U = e^{-i H \Delta t}, one matrix exponential per time step.   (T, 2, 2) → (T, 2, 2)"""
    return torch.linalg.matrix_exp(-1j * H * dt)


def step(U: torch.Tensor, psi: torch.Tensor) -> torch.Tensor:
    r"""\psi_a \leftarrow U_{ab} \psi_b.   U: (2, 2), psi: (2,) → (2,)"""
    return torch.einsum("ab,b->a", U, psi)


def bloch_vector(psi: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    r"""s_k = \psi^*_a \sigma_{kab} \psi_b.   psi: (2,) → (3,) real"""
    return torch.einsum("a,kab,b->k", psi.conj(), sigma, psi).real


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/spin_precession/tensors.py
"""Build the concrete tensors: Pauli tensor, Levi-Civita symbol, the field history and the initial spinor."""

import math

import torch


def pauli(device: torch.device) -> torch.Tensor:
    """σ_{kab}: the three Pauli matrices stacked along k. Shape (3, 2, 2) complex."""
    sx = [[0, 1], [1, 0]]
    sy = [[0, -1j], [1j, 0]]
    sz = [[1, 0], [0, -1]]
    return torch.tensor([sx, sy, sz], dtype=torch.complex128, device=device)


def levi_civita(device: torch.device) -> torch.Tensor:
    """ε_{ijk}: +1 for even permutations of (0, 1, 2), −1 for odd ones, 0 otherwise. Shape (3, 3, 3)."""
    eps = torch.zeros(3, 3, 3, dtype=torch.complex128, device=device)
    for i, j, k in [(0, 1, 2), (1, 2, 0), (2, 0, 1)]:
        eps[i, j, k] = 1.0
        eps[j, i, k] = -1.0
    return eps


def field(times: torch.Tensor, B0: float, B1: float, drive_freq: float) -> torch.Tensor:
    """B_{tk} = (B1 cos ωt, 0, B0) at every time: static field along z plus a linear drive along x. Shape (T, 3)."""
    bx = B1 * torch.cos(drive_freq * times)  # (T,)
    by = torch.zeros_like(times)
    bz = torch.full_like(times, B0)
    return torch.stack([bx, by, bz], dim=-1)


def tilted_spinor(theta_deg: float, device: torch.device) -> torch.Tensor:
    """ψ_a = (cos θ/2, sin θ/2): spin pointing at angle θ from +z in the xz-plane. Shape (2,) complex."""
    half = math.radians(theta_deg) / 2
    return torch.tensor([math.cos(half), math.sin(half)], dtype=torch.complex128, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/spin_precession/solve.py
"""Evolve the spinor step by step with the exact one-step propagators, recording the Bloch vector."""

import torch

from . import tensors
from .equation import bloch_vector, hamiltonian, propagator, step


def solve(
    psi0: torch.Tensor,
    sigma: torch.Tensor,
    gamma: float,
    B0: float,
    B1: float,
    drive_freq: float,
    t_max: float,
    dt: float = 0.02,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, 2) complex, s (T, 3)).

    All T−1 propagators are built in one batch: the field at each step's midpoint → H (T−1, 2, 2) → U.
    Only the product U_{n} ⋯ U_1 ψ0 has to run as a loop, because each step needs the previous state.
    """
    n_steps = int(round(t_max / dt))
    times = dt * torch.arange(n_steps + 1, dtype=torch.float64, device=psi0.device)  # (T,)
    B = tensors.field(times[:-1] + dt / 2, B0, B1, drive_freq)  # (T-1, 3) midpoint rule
    U = propagator(hamiltonian(B, sigma, gamma), dt)  # (T-1, 2, 2)

    states = [psi0]
    for n in range(n_steps):
        states.append(step(U[n], states[-1]))
    psi = torch.stack(states)  # (T, 2)
    s = torch.stack([bloch_vector(p, sigma) for p in psi])  # (T, 3)
    return times, psi, s


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/spin_precession/plot.py
"""Animate the Bloch vector on the Bloch sphere (left) and its components s_k(t) (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def draw_sphere(ax) -> None:
    """Faint unit sphere with equator, a meridian and the x, y, z axes."""
    u = np.linspace(0, 2 * np.pi, 60)
    ax.plot(np.cos(u), np.sin(u), 0 * u, color="#9ca3af", lw=0.7)  # equator
    ax.plot(np.cos(u), 0 * u, np.sin(u), color="#d1d5db", lw=0.7)  # xz meridian
    ax.plot(0 * u, np.cos(u), np.sin(u), color="#d1d5db", lw=0.7)  # yz meridian
    for axis, name in zip(np.eye(3), "xyz"):
        ax.plot(*np.stack([-axis, axis], axis=1), color="#9ca3af", lw=0.6, ls=":")
        ax.text(*(1.2 * axis), name, color="#4b5563", ha="center", va="center")
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    ax.view_init(elev=20, azim=35)


def animate(times: torch.Tensor, s: torch.Tensor, path: Path) -> Path:
    s = s.cpu().numpy()  # (T, 3)
    t = times.cpu().numpy()

    fig = plt.figure(figsize=(7, 4))
    ax3 = fig.add_subplot(1, 2, 1, projection="3d")
    draw_sphere(ax3)
    (trail,) = ax3.plot([], [], [], color=ACCENT[0], lw=1, alpha=0.6)
    (arrow,) = ax3.plot([], [], [], color=ACCENT[1], lw=2.5)
    (tip,) = ax3.plot([], [], [], "o", color=ACCENT[1], ms=5)

    ax = fig.add_subplot(1, 2, 2)
    ax.set_xlim(t[0], t[-1])
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel("t")
    ax.set_ylabel("Bloch vector s_k")
    curves = [ax.plot([], [], lw=1.4, label=f"s_{k}")[0] for k in "xyz"]
    ax.legend(loc="lower left", fontsize=8, ncol=3)
    clock = ax.set_title("t = 0")  # placeholder so tight_layout leaves room

    def update(step: int):
        trail.set_data_3d(s[: step + 1, 0], s[: step + 1, 1], s[: step + 1, 2])
        arrow.set_data_3d([0, s[step, 0]], [0, s[step, 1]], [0, s[step, 2]])
        tip.set_data_3d([s[step, 0]], [s[step, 1]], [s[step, 2]])
        for k, curve in enumerate(curves):
            curve.set_data(t[: step + 1], s[: step + 1, k])
        clock.set_text(f"t = {t[step]:.1f}   |s| = {np.linalg.norm(s[step]):.4f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'B0': 1.0,
    'B1': 0.2,
    'drive_freq': 1.0,
    'gamma': 1.0,
    'theta0': 20.0,
    't_max': 32.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.spin_precession")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.spin_precession.main")
problem_dir = Path("problems/quantum/spin_precession")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
